In [ ]:
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "dataset_config.py").exists())))
from dataset_config import RAW_DIR, FINAL_DIR

### "../All_Datasets/Plant_leaf_diseases_dataset_with_augmentation/Plant_leave_diseases_dataset_with_augmentation"

### 19/02/25

### PlantVillage Dataset

#### Category wise balancing

In [4]:
import os
import random
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# Set the paths to your original PlantVillage dataset and the balanced output directory.
base_dir = str(RAW_DIR / "plantvillage_raw")      # original dataset folder (each subfolder = a category)
balanced_dir = str(FINAL_DIR / "plantvillage_balanced")        # output folder with balanced categories

# Define the target number of images per category.
# You can adjust this value so that you do not increase the dataset too much.
TARGET_COUNT = 2000

# Define augmentation parameters.
datagen = ImageDataGenerator(
    rotation_range=40,          # rotate images up to 40 degrees
    width_shift_range=0.2,      # shift images horizontally by up to 20%
    height_shift_range=0.2,     # shift images vertically by up to 20%
    shear_range=0.2,            # apply shear transformation
    zoom_range=0.2,             # zoom in/out by up to 20%
    horizontal_flip=True,       # randomly flip images
    fill_mode='nearest'         # fill in newly created pixels
)

# Make sure the output directory exists
os.makedirs(balanced_dir, exist_ok=True)

# Process each category folder.
for category in os.listdir(base_dir):
    category_path = os.path.join(base_dir, category)
    if not os.path.isdir(category_path):
        continue  # skip files—only process directories

    print(f'Processing category: {category}')
    
    # Get a list of image filenames (adjust extensions as needed)
    images = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    count = len(images)

    # Create the corresponding folder in the balanced dataset.
    target_category_dir = os.path.join(balanced_dir, category)
    os.makedirs(target_category_dir, exist_ok=True)

    if count >= TARGET_COUNT:
        # If there are too many images, randomly select TARGET_COUNT images.
        selected_images = random.sample(images, TARGET_COUNT)
        for img_name in selected_images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)
        print(f'  - Kept {TARGET_COUNT} images (undersampled from {count}).')
    else:
        # Copy the original images first.
        for img_name in images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)

        # Determine how many augmented images are needed.
        num_to_generate = TARGET_COUNT - count
        print(f'  - Augmenting {num_to_generate} images to reach {TARGET_COUNT} total.')

        i = 0
        # Use a round-robin approach: for each original image, generate one augmented image
        # until the target is reached.
        while i < num_to_generate:
            for img_name in images:
                if i >= num_to_generate:
                    break

                img_path = os.path.join(category_path, img_name)
                # Load the image and convert it to an array.
                image = load_img(img_path)
                x = img_to_array(image)
                x = x.reshape((1,) + x.shape)

                # Generate one augmented image.
                aug_iter = datagen.flow(x, batch_size=1)
                aug_image = next(aug_iter)[0].astype('uint8')

                # Create a new filename for the augmented image.
                base_name, ext = os.path.splitext(img_name)
                new_filename = f"{base_name}_aug_{i}{ext}"
                save_path = os.path.join(target_category_dir, new_filename)

                # Save the augmented image.
                save_img(save_path, aug_image)
                i += 1

        print(f'  - Category "{category}" now has {TARGET_COUNT} images.')


Processing category: Apple___Apple_scab
  - Augmenting 1000 images to reach 2000 total.
  - Category "Apple___Apple_scab" now has 2000 images.
Processing category: Apple___Black_rot
  - Augmenting 1000 images to reach 2000 total.
  - Category "Apple___Black_rot" now has 2000 images.
Processing category: Apple___Cedar_apple_rust
  - Augmenting 1000 images to reach 2000 total.
  - Category "Apple___Cedar_apple_rust" now has 2000 images.
Processing category: Apple___healthy
  - Augmenting 355 images to reach 2000 total.
  - Category "Apple___healthy" now has 2000 images.
Processing category: Background_without_leaves
  - Augmenting 857 images to reach 2000 total.
  - Category "Background_without_leaves" now has 2000 images.
Processing category: Blueberry___healthy
  - Augmenting 498 images to reach 2000 total.
  - Category "Blueberry___healthy" now has 2000 images.
Processing category: Cherry___healthy
  - Augmenting 1000 images to reach 2000 total.
  - Category "Cherry___healthy" now has

#### Plant wise balancing

In [8]:
import os
import random
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# === CONFIGURATION ===
# Paths to your original dataset and the output directory.
base_dir = str(RAW_DIR / "plantvillage_raw")  # original dataset (each subfolder = a category, e.g. "Apple___healthy")
balanced_dir = str(FINAL_DIR / "plantvillage_balanced_by_plant")          # output directory (folders will have the same names)

# Set a fixed total number of images per plant.
GLOBAL_PLANT_TARGET = 5000  # Total images for each plant after balancing

# Augmentation parameters – adjust as needed.
datagen = ImageDataGenerator(
    rotation_range=40,          
    width_shift_range=0.2,      
    height_shift_range=0.2,     
    shear_range=0.2,            
    zoom_range=0.2,             
    horizontal_flip=True,       
    fill_mode='nearest'         
)

# === STEP 1: Group images by plant.
# We assume each folder is named like "PlantName___Disease" (e.g., "Apple___healthy").
# Build a dictionary: plant -> { folder_name: [(full_image_path, filename), ...] }
plant_groups = {}

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if not os.path.isdir(folder_path):
        continue  # Skip files.
    # Extract plant name (everything before "___")
    plant = folder.split("___")[0]
    if plant not in plant_groups:
        plant_groups[plant] = {}
    # List image files (adjust extensions if needed)
    images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    # Save full path info along with the filename
    plant_groups[plant][folder] = [(os.path.join(folder_path, f), f) for f in images]

# === STEP 2: Determine target counts per plant and per folder.
# For plants with both healthy and disease folders:
#   - We'll set the healthy target to half of GLOBAL_PLANT_TARGET.
#   - The disease images (total) will be the remaining half, split equally among all disease folders.
# For plants with only one folder, that folder’s target is GLOBAL_PLANT_TARGET.
for plant, folders_dict in plant_groups.items():
    print(f"\nProcessing plant: {plant}")
    
    healthy_key = None
    disease_keys = []
    for folder_name in folders_dict:
        if folder_name.lower().endswith("___healthy"):
            healthy_key = folder_name
        else:
            disease_keys.append(folder_name)
    
    if healthy_key and disease_keys:
        # Both healthy and disease folders exist.
        target_healthy = GLOBAL_PLANT_TARGET // 2  # e.g., 1500 images for healthy
        total_disease_target = GLOBAL_PLANT_TARGET - target_healthy
        num_disease = len(disease_keys)
        # Each disease folder gets an equal share.
        target_disease = total_disease_target // num_disease
        print(f"  Found healthy folder '{healthy_key}' and {len(disease_keys)} disease folders.")
        print(f"  => Setting target: healthy = {target_healthy}, each disease folder = {target_disease}")
    elif healthy_key:
        # Only healthy folder exists.
        target_healthy = GLOBAL_PLANT_TARGET
        print(f"  Only healthy folder '{healthy_key}' exists. Target = {target_healthy}")
    elif disease_keys:
        # Only disease folders exist.
        target_disease = GLOBAL_PLANT_TARGET
        print(f"  Only disease folders exist. Target per disease folder = {target_disease}")
    else:
        print(f"  No images found for plant {plant}.")
        continue

    # === STEP 3: Process (balance) each folder for this plant.
    # The output folders will keep the same names as the original.
    for folder_name, img_info_list in folders_dict.items():
        # Decide the target for this folder.
        if healthy_key and disease_keys:
            if folder_name == healthy_key:
                target = target_healthy
            else:
                target = target_disease
        else:
            target = GLOBAL_PLANT_TARGET  # Only one type exists.
        
        print(f"  Processing folder '{folder_name}': current count = {len(img_info_list)}; target = {target}")

        # Create the corresponding output folder.
        out_folder = os.path.join(balanced_dir, folder_name)
        os.makedirs(out_folder, exist_ok=True)

        if len(img_info_list) == 0:
            continue

        # --- Undersampling if folder has more images than target.
        if len(img_info_list) > target:
            selected = random.sample(img_info_list, target)
            for src_path, filename in selected:
                dst_path = os.path.join(out_folder, filename)
                shutil.copy2(src_path, dst_path)
            print(f"    - Undersampled to {target} images.")
        else:
            # --- First, copy all original images.
            for src_path, filename in img_info_list:
                dst_path = os.path.join(out_folder, filename)
                shutil.copy2(src_path, dst_path)
            # --- Augment until reaching the target.
            num_to_generate = target - len(img_info_list)
            print(f"    - Augmenting with {num_to_generate} images.")
            gen_count = 0
            # Use a round-robin strategy over the original images.
            while gen_count < num_to_generate:
                for src_path, filename in img_info_list:
                    if gen_count >= num_to_generate:
                        break
                    image = load_img(src_path)
                    x = img_to_array(image)
                    x = x.reshape((1,) + x.shape)
                    aug_iter = datagen.flow(x, batch_size=1)
                    aug_image = next(aug_iter)[0].astype('uint8')
                    base_name, ext = os.path.splitext(filename)
                    new_filename = f"{base_name}_aug_{gen_count}{ext}"
                    save_path = os.path.join(out_folder, new_filename)
                    save_img(save_path, aug_image)
                    gen_count += 1
            print(f"    - Finished with {target} images in folder '{folder_name}'.")



Processing plant: Apple
  Found healthy folder 'Apple___healthy' and 3 disease folders.
  => Setting target: healthy = 2500, each disease folder = 833
  Processing folder 'Apple___Apple_scab': current count = 1000; target = 833
    - Undersampled to 833 images.
  Processing folder 'Apple___Black_rot': current count = 1000; target = 833
    - Undersampled to 833 images.
  Processing folder 'Apple___Cedar_apple_rust': current count = 1000; target = 833
    - Undersampled to 833 images.
  Processing folder 'Apple___healthy': current count = 1645; target = 2500
    - Augmenting with 855 images.
    - Finished with 2500 images in folder 'Apple___healthy'.

Processing plant: Background_without_leaves
  Only disease folders exist. Target per disease folder = 5000
  Processing folder 'Background_without_leaves': current count = 1143; target = 5000
    - Augmenting with 3857 images.
    - Finished with 5000 images in folder 'Background_without_leaves'.

Processing plant: Blueberry
  Only healt

### New Plant Disease Dataset

In [2]:
import os
import random
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# Set the paths to your original PlantVillage dataset and the balanced output directory.
base_dir = str(RAW_DIR / "new_plant_diseases_raw")      # original dataset folder (each subfolder = a category)
balanced_dir = str(FINAL_DIR / "new_plant_diseases_balanced")        # output folder with balanced categories

# Define the target number of images per category.
# You can adjust this value so that you do not increase the dataset too much.
TARGET_COUNT = 2300

# Define augmentation parameters.
datagen = ImageDataGenerator(
    rotation_range=40,          # rotate images up to 40 degrees
    width_shift_range=0.2,      # shift images horizontally by up to 20%
    height_shift_range=0.2,     # shift images vertically by up to 20%
    shear_range=0.2,            # apply shear transformation
    zoom_range=0.2,             # zoom in/out by up to 20%
    horizontal_flip=True,       # randomly flip images
    fill_mode='nearest'         # fill in newly created pixels
)

# Make sure the output directory exists
os.makedirs(balanced_dir, exist_ok=True)

# Process each category folder.
for category in os.listdir(base_dir):
    category_path = os.path.join(base_dir, category)
    if not os.path.isdir(category_path):
        continue  # skip files—only process directories

    print(f'Processing category: {category}')
    
    # Get a list of image filenames (adjust extensions as needed)
    images = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    count = len(images)

    # Create the corresponding folder in the balanced dataset.
    target_category_dir = os.path.join(balanced_dir, category)
    os.makedirs(target_category_dir, exist_ok=True)

    if count >= TARGET_COUNT:
        # If there are too many images, randomly select TARGET_COUNT images.
        selected_images = random.sample(images, TARGET_COUNT)
        for img_name in selected_images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)
        print(f'  - Kept {TARGET_COUNT} images (undersampled from {count}).')
    else:
        # Copy the original images first.
        for img_name in images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)

        # Determine how many augmented images are needed.
        num_to_generate = TARGET_COUNT - count
        print(f'  - Augmenting {num_to_generate} images to reach {TARGET_COUNT} total.')

        i = 0
        # Use a round-robin approach: for each original image, generate one augmented image
        # until the target is reached.
        while i < num_to_generate:
            for img_name in images:
                if i >= num_to_generate:
                    break

                img_path = os.path.join(category_path, img_name)
                # Load the image and convert it to an array.
                image = load_img(img_path)
                x = img_to_array(image)
                x = x.reshape((1,) + x.shape)

                # Generate one augmented image.
                aug_iter = datagen.flow(x, batch_size=1)
                aug_image = next(aug_iter)[0].astype('uint8')

                # Create a new filename for the augmented image.
                base_name, ext = os.path.splitext(img_name)
                new_filename = f"{base_name}_aug_{i}{ext}"
                save_path = os.path.join(target_category_dir, new_filename)

                # Save the augmented image.
                save_img(save_path, aug_image)
                i += 1

        print(f'  - Category "{category}" now has {TARGET_COUNT} images.')


Processing category: Apple___Apple_scab
  - Kept 2300 images (undersampled from 2523).
Processing category: Apple___Black_rot
  - Kept 2300 images (undersampled from 2484).
Processing category: Apple___Cedar_apple_rust
  - Augmenting 96 images to reach 2300 total.
  - Category "Apple___Cedar_apple_rust" now has 2300 images.
Processing category: Apple___healthy
  - Kept 2300 images (undersampled from 2510).
Processing category: Blueberry___healthy
  - Augmenting 30 images to reach 2300 total.
  - Category "Blueberry___healthy" now has 2300 images.
Processing category: Cherry_(including_sour)___healthy
  - Augmenting 18 images to reach 2300 total.
  - Category "Cherry_(including_sour)___healthy" now has 2300 images.
Processing category: Cherry_(including_sour)___Powdery_mildew
  - Augmenting 196 images to reach 2300 total.
  - Category "Cherry_(including_sour)___Powdery_mildew" now has 2300 images.
Processing category: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  - Augmenting 248

### Combining the train and valid folders

In [1]:
import os
import random
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# Set the paths to your original PlantVillage dataset and the balanced output directory.
base_dir = "../All_Datasets/New Plant Diseases Dataset(Augmented)/train"      # original dataset folder (each subfolder = a category)
balanced_dir = "../All_Datasets/New Plant Diseases Dataset (Balanced)/train"        # output folder with balanced categories

# Define the target number of images per category.
# You can adjust this value so that you do not increase the dataset too much.
TARGET_COUNT = 2000

# Define augmentation parameters.
datagen = ImageDataGenerator(
    rotation_range=40,          # rotate images up to 40 degrees
    width_shift_range=0.2,      # shift images horizontally by up to 20%
    height_shift_range=0.2,     # shift images vertically by up to 20%
    shear_range=0.2,            # apply shear transformation
    zoom_range=0.2,             # zoom in/out by up to 20%
    horizontal_flip=True,       # randomly flip images
    fill_mode='nearest'         # fill in newly created pixels
)

# Make sure the output directory exists
os.makedirs(balanced_dir, exist_ok=True)

# Process each category folder.
for category in os.listdir(base_dir):
    category_path = os.path.join(base_dir, category)
    if not os.path.isdir(category_path):
        continue  # skip files—only process directories

    print(f'Processing category: {category}')
    
    # Get a list of image filenames (adjust extensions as needed)
    images = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    count = len(images)

    # Create the corresponding folder in the balanced dataset.
    target_category_dir = os.path.join(balanced_dir, category)
    os.makedirs(target_category_dir, exist_ok=True)

    if count >= TARGET_COUNT:
        # If there are too many images, randomly select TARGET_COUNT images.
        selected_images = random.sample(images, TARGET_COUNT)
        for img_name in selected_images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)
        print(f'  - Kept {TARGET_COUNT} images (undersampled from {count}).')
    else:
        # Copy the original images first.
        for img_name in images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)

        # Determine how many augmented images are needed.
        num_to_generate = TARGET_COUNT - count
        print(f'  - Augmenting {num_to_generate} images to reach {TARGET_COUNT} total.')

        i = 0
        # Use a round-robin approach: for each original image, generate one augmented image
        # until the target is reached.
        while i < num_to_generate:
            for img_name in images:
                if i >= num_to_generate:
                    break

                img_path = os.path.join(category_path, img_name)
                # Load the image and convert it to an array.
                image = load_img(img_path)
                x = img_to_array(image)
                x = x.reshape((1,) + x.shape)

                # Generate one augmented image.
                aug_iter = datagen.flow(x, batch_size=1)
                aug_image = next(aug_iter)[0].astype('uint8')

                # Create a new filename for the augmented image.
                base_name, ext = os.path.splitext(img_name)
                new_filename = f"{base_name}_aug_{i}{ext}"
                save_path = os.path.join(target_category_dir, new_filename)

                # Save the augmented image.
                save_img(save_path, aug_image)
                i += 1

        print(f'  - Category "{category}" now has {TARGET_COUNT} images.')


Processing category: Apple___Apple_scab
  - Kept 2000 images (undersampled from 2016).
Processing category: Apple___Black_rot
  - Augmenting 13 images to reach 2000 total.
  - Category "Apple___Black_rot" now has 2000 images.
Processing category: Apple___Cedar_apple_rust
  - Augmenting 240 images to reach 2000 total.
  - Category "Apple___Cedar_apple_rust" now has 2000 images.
Processing category: Apple___healthy
  - Kept 2000 images (undersampled from 2008).
Processing category: Blueberry___healthy
  - Augmenting 184 images to reach 2000 total.
  - Category "Blueberry___healthy" now has 2000 images.
Processing category: Cherry_(including_sour)___healthy
  - Augmenting 174 images to reach 2000 total.
  - Category "Cherry_(including_sour)___healthy" now has 2000 images.
Processing category: Cherry_(including_sour)___Powdery_mildew
  - Augmenting 317 images to reach 2000 total.
  - Category "Cherry_(including_sour)___Powdery_mildew" now has 2000 images.
Processing category: Corn_(maize)_

In [2]:
import os
import random
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, save_img

# Set the paths to your original PlantVillage dataset and the balanced output directory.
base_dir = "../All_Datasets/New Plant Diseases Dataset(Augmented)/valid"      # original dataset folder (each subfolder = a category)
balanced_dir = "../All_Datasets/New Plant Diseases Dataset (Balanced)/valid"        # output folder with balanced categories

# Define the target number of images per category.
# You can adjust this value so that you do not increase the dataset too much.
TARGET_COUNT = 2000

# Define augmentation parameters.
datagen = ImageDataGenerator(
    rotation_range=40,          # rotate images up to 40 degrees
    width_shift_range=0.2,      # shift images horizontally by up to 20%
    height_shift_range=0.2,     # shift images vertically by up to 20%
    shear_range=0.2,            # apply shear transformation
    zoom_range=0.2,             # zoom in/out by up to 20%
    horizontal_flip=True,       # randomly flip images
    fill_mode='nearest'         # fill in newly created pixels
)

# Make sure the output directory exists
os.makedirs(balanced_dir, exist_ok=True)

# Process each category folder.
for category in os.listdir(base_dir):
    category_path = os.path.join(base_dir, category)
    if not os.path.isdir(category_path):
        continue  # skip files—only process directories

    print(f'Processing category: {category}')
    
    # Get a list of image filenames (adjust extensions as needed)
    images = [f for f in os.listdir(category_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    count = len(images)

    # Create the corresponding folder in the balanced dataset.
    target_category_dir = os.path.join(balanced_dir, category)
    os.makedirs(target_category_dir, exist_ok=True)

    if count >= TARGET_COUNT:
        # If there are too many images, randomly select TARGET_COUNT images.
        selected_images = random.sample(images, TARGET_COUNT)
        for img_name in selected_images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)
        print(f'  - Kept {TARGET_COUNT} images (undersampled from {count}).')
    else:
        # Copy the original images first.
        for img_name in images:
            src = os.path.join(category_path, img_name)
            dst = os.path.join(target_category_dir, img_name)
            shutil.copy2(src, dst)

        # Determine how many augmented images are needed.
        num_to_generate = TARGET_COUNT - count
        print(f'  - Augmenting {num_to_generate} images to reach {TARGET_COUNT} total.')

        i = 0
        # Use a round-robin approach: for each original image, generate one augmented image
        # until the target is reached.
        while i < num_to_generate:
            for img_name in images:
                if i >= num_to_generate:
                    break

                img_path = os.path.join(category_path, img_name)
                # Load the image and convert it to an array.
                image = load_img(img_path)
                x = img_to_array(image)
                x = x.reshape((1,) + x.shape)

                # Generate one augmented image.
                aug_iter = datagen.flow(x, batch_size=1)
                aug_image = next(aug_iter)[0].astype('uint8')

                # Create a new filename for the augmented image.
                base_name, ext = os.path.splitext(img_name)
                new_filename = f"{base_name}_aug_{i}{ext}"
                save_path = os.path.join(target_category_dir, new_filename)

                # Save the augmented image.
                save_img(save_path, aug_image)
                i += 1

        print(f'  - Category "{category}" now has {TARGET_COUNT} images.')


Processing category: Apple___Apple_scab
  - Augmenting 1496 images to reach 2000 total.
  - Category "Apple___Apple_scab" now has 2000 images.
Processing category: Apple___Black_rot
  - Augmenting 1503 images to reach 2000 total.
  - Category "Apple___Black_rot" now has 2000 images.
Processing category: Apple___Cedar_apple_rust
  - Augmenting 1560 images to reach 2000 total.
  - Category "Apple___Cedar_apple_rust" now has 2000 images.
Processing category: Apple___healthy
  - Augmenting 1498 images to reach 2000 total.
  - Category "Apple___healthy" now has 2000 images.
Processing category: Blueberry___healthy
  - Augmenting 1546 images to reach 2000 total.
  - Category "Blueberry___healthy" now has 2000 images.
Processing category: Cherry_(including_sour)___healthy
  - Augmenting 1544 images to reach 2000 total.
  - Category "Cherry_(including_sour)___healthy" now has 2000 images.
Processing category: Cherry_(including_sour)___Powdery_mildew
  - Augmenting 1579 images to reach 2000 tot

In [6]:
import os
import shutil

# Define source directories and destination directory.
dir_valid = "../All_Datasets/New Plant Diseases Dataset (Balanced)/valid"
dir_train = "../All_Datasets/New Plant Diseases Dataset (Balanced)/train"
dest_dir = "../All_Datasets/New Plant Diseases Dataset (Balanced)/combine"

# Create destination directory if it doesn't exist.
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

# Use the train folder to determine the list of categories (assumed to be the same in valid).
categories = sorted([d for d in os.listdir(dir_train) if os.path.isdir(os.path.join(dir_train, d))])
print("Found categories:", categories)

# Process each category.
for category in categories:
    src_train_cat = os.path.join(dir_train, category)
    src_valid_cat = os.path.join(dir_valid, category)
    dest_cat = os.path.join(dest_dir, category)
    
    # Create the destination subfolder.
    if not os.path.exists(dest_cat):
        os.makedirs(dest_cat)
    
    # Initialize counter.
    count = 1
    
    # Copy images from train folder.
    if os.path.exists(src_train_cat):
        train_files = sorted(os.listdir(src_train_cat))
        for filename in train_files:
            src_file = os.path.join(src_train_cat, filename)
            # Extract the file extension.
            _, ext = os.path.splitext(filename)
            new_filename = f"image{count}{ext}"
            dest_file = os.path.join(dest_cat, new_filename)
            try:
                shutil.copy2(src_file, dest_file)
            except Exception as e:
                print(f"Error copying {src_file} to {dest_file}: {e}")
            count += 1

    # Copy images from valid folder.
    if os.path.exists(src_valid_cat):
        valid_files = sorted(os.listdir(src_valid_cat))
        for filename in valid_files:
            src_file = os.path.join(src_valid_cat, filename)
            _, ext = os.path.splitext(filename)
            new_filename = f"image{count}{ext}"
            dest_file = os.path.join(dest_cat, new_filename)
            try:
                shutil.copy2(src_file, dest_file)
            except Exception as e:
                print(f"Error copying {src_file} to {dest_file}: {e}")
            count += 1
    
    print(f"Category: {category} now has {count - 1} images in the combine folder.")


Found categories: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', '

### Combining the Unbalanced train and valid folders of New Plant Disease.

In [1]:
import os
import shutil

# Define source directories and destination directory.
dir_valid = "../All_Datasets/New Plant Diseases Dataset(Augmented)/valid"
dir_train = "../All_Datasets/New Plant Diseases Dataset(Augmented)/train"
dest_dir = "../All_Datasets/New Plant Diseases Dataset(Unbalanced)"

# Create destination directory if it doesn't exist.
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

# Use the train folder to determine the list of categories (assumed to be the same in valid).
categories = sorted([d for d in os.listdir(dir_train) if os.path.isdir(os.path.join(dir_train, d))])
print("Found categories:", categories)

# Process each category.
for category in categories:
    src_train_cat = os.path.join(dir_train, category)
    src_valid_cat = os.path.join(dir_valid, category)
    dest_cat = os.path.join(dest_dir, category)
    
    # Create the destination subfolder.
    if not os.path.exists(dest_cat):
        os.makedirs(dest_cat)
    
    # Initialize counter.
    count = 1
    
    # Copy images from train folder.
    if os.path.exists(src_train_cat):
        train_files = sorted(os.listdir(src_train_cat))
        for filename in train_files:
            src_file = os.path.join(src_train_cat, filename)
            # Extract the file extension.
            _, ext = os.path.splitext(filename)
            new_filename = f"image{count}{ext}"
            dest_file = os.path.join(dest_cat, new_filename)
            try:
                shutil.copy2(src_file, dest_file)
            except Exception as e:
                print(f"Error copying {src_file} to {dest_file}: {e}")
            count += 1

    # Copy images from valid folder.
    if os.path.exists(src_valid_cat):
        valid_files = sorted(os.listdir(src_valid_cat))
        for filename in valid_files:
            src_file = os.path.join(src_valid_cat, filename)
            _, ext = os.path.splitext(filename)
            new_filename = f"image{count}{ext}"
            dest_file = os.path.join(dest_cat, new_filename)
            try:
                shutil.copy2(src_file, dest_file)
            except Exception as e:
                print(f"Error copying {src_file} to {dest_file}: {e}")
            count += 1
    
    print(f"Category: {category} now has {count - 1} images in the combine folder.")


Found categories: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', '